<a href="https://colab.research.google.com/github/Musamehar/ML_Intership/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Chosen Lane: Lane 1 — Refresh / Content Opportunity Scoring

Task Type: Learning-to-Rank / Priority Scoring (Supervised Ranking)

Why Ranking / Scoring?
We are not simply asking "Will this page decline?" (Binary Classification). Instead, our user (an SEO manager or editor) has limited bandwidth for example, capacity to audit 20 pages per week out of thousands. Our task is to output a continuous risk score that ranks the entire inventory so the highest-risk, highest-value pages float to the top of the queue.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Starter Proxy Label: is_declining_label (derived from trend_direction == 'down' in the trailing window).

Ideal Capstone Target: Future 30-day traffic decay (clicks_future_30d < 0.80 * clicks_prior_30d), observed in a subsequent evaluation window.

Leakage Avoidance: We explicitly exclude trend_pct and trend_direction from the feature set during training to prevent target leakage.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Metric: Precision@20 (and Precision@50).Why Precision@K? In a human-in-the-loop workflow, an editor reviews the top $K$ items in the generated queue. Precision@$K$ directly measures the percentage of those top $K$ flagged items that actually represented true performance decay. Generic accuracy or ROC-AUC would evaluate the entire long tail of thousands of low-traffic pages, which is irrelevant to the actual business decision.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import os
import pandas as pd
from pathlib import Path

if not os.path.exists('ML_Intership') and not Path('data/raw/content_refresh_anonymized.csv').exists():
    !git clone https://github.com/Musamehar/ML_Intership.git

possible_paths = [
    Path('ML_Intership/data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv')
]

data_path = next((p for p in possible_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv.")

df = pd.read_csv(data_path)

df_clean = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id').copy()

if 'is_declining_label' in df_clean.columns:
    df_clean['target_decay'] = df_clean['is_declining_label'].astype(int)
else:
    df_clean['target_decay'] = (df_clean['trend_direction'] == 'down').astype(int)

display_cols = ['content_id', 'client_id', 'content_type', 'impressions_90d', 'clicks_90d', 'avg_position', 'target_decay']
print(f"Unit of Analysis: 1 Row = 1 Content Item ('content_id') for a specific Client ('client_id')")
print(f"Dataframe Shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns\n")

df_clean[display_cols].head(5)

Cloning into 'ML_Intership'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 109 (delta 26), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 1.84 MiB | 9.67 MiB/s, done.
Resolving deltas: 100% (26/26), done.
Unit of Analysis: 1 Row = 1 Content Item ('content_id') for a specific Client ('client_id')
Dataframe Shape: 30,000 rows x 45 columns



,content_id,client_id,content_type,impressions_90d,clicks_90d,avg_position,target_decay
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,10.6,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,20.3,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,36.5,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,6.2,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,44.0,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A static heuristic rule (such as if content_age > 180 and trend == 'down') suffers from three critical flaws:

Rigid Cutoffs: Hard thresholds fail to capture non-linear relationships—for instance, a 5% drop on a Page 1 high-traffic URL is far more critical than a 30% drop on a Page 5 low-volume URL.

Signal Tangling: Decay is driven by multi-variable interaction across search volume, CTR volatility, intent match, freshness, and position drift. Manual rules cannot weight these dynamically across different content_type categories.

No Prioritization: A fixed rule flags hundreds of pages as a binary "YES", forcing human editors to manually sort through them. Machine learning scores candidates on a continuous scale, creating an actionable priority queue that optimizes limited human effort.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.